In [1]:
import subprocess
subprocess.run(["pip","install","cdsapi","xarray","netCDF4","pandas","matplotlib"], check=True)
print("Done!")

Done!


## Configuration

In [2]:
import os
lat           = 34.499984
lon           = -4.708586
site_name     = "test_site"
startDate     = "2005-01-01"
endDate       = "2024-12-31"
selected_year = 2015
print("lat:", lat, "lon:", lon)
print("Period:", startDate, "to", endDate)

lat: 34.499984 lon: -4.708586
Period: 2005-01-01 to 2024-12-31


## Step 1 — Download ERA5
Make sure your ~/.cdsapirc file has your Copernicus API key.

In [3]:
import cdsapi

output_file = site_name + "_era5_temperature.nc"

if os.path.exists(output_file):
    print("File exists, skipping download:", output_file)
else:
    print("Connecting to Copernicus CDS...")
    start_year = int(startDate[:4])
    end_year   = int(endDate[:4])
    years  = [str(y) for y in range(start_year, end_year + 1)]
    months = [str(m).zfill(2) for m in range(1, 13)]
    days   = [str(d).zfill(2) for d in range(1, 32)]
    area   = [lat + 0.25, lon - 0.25, lat - 0.25, lon + 0.25]
    c = cdsapi.Client()
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable":     "2m_temperature",
            "year":         years,
            "month":        months,
            "day":          days,
            "time":         ["00:00", "06:00", "12:00", "18:00"],
            "area":         area,
            "data_format":  "netcdf",
        },
        output_file
    )
    print("Download complete:", output_file)

Connecting to Copernicus CDS...


HTTPError: 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-single-levels/execution
cost limits exceeded
Your request is too large, please reduce your selection.

## Step 2 — Process into daily DataFrame

In [ ]:
import xarray as xr
import pandas as pd

ds   = xr.open_dataset(output_file)
temp = ds["t2m"]
tp   = temp.sel(latitude=lat, longitude=lon, method="nearest")
df   = tp.to_dataframe(name="tk").reset_index()
df["temperature"] = df["tk"] - 273.15
tcol = "valid_time" if "valid_time" in df.columns else "time"
df["date"] = pd.to_datetime(df[tcol])
df = df[["date","temperature"]].copy()
df = df[(df["date"] >= startDate) & (df["date"] <= endDate)]
df["date"] = df["date"].dt.date
daily = df.groupby("date")["temperature"].mean().reset_index()
daily.columns = ["date","temperature"]
daily["date"] = pd.to_datetime(daily["date"])
print("Total days:", len(daily))
print("Min temp:", round(daily["temperature"].min(), 2), "C")
print("Max temp:", round(daily["temperature"].max(), 2), "C")
daily.head(10)

## Step 3 — Analysis: best, worst, selected year

In [ ]:
daily["year"] = daily["date"].dt.year
yearly = daily.groupby("year")["temperature"].mean()

best_year  = int(yearly.idxmax())
worst_year = int(yearly.idxmin())

print("=" * 45)
print("Best year  (hottest):", best_year, "avg", round(yearly[best_year], 2), "C")
print("Worst year (coldest):", worst_year, "avg", round(yearly[worst_year], 2), "C")

if selected_year in yearly.index:
    sel = daily[daily["year"] == selected_year]
    print("
Selected year:", selected_year)
    print("  Average    :", round(yearly[selected_year], 2), "C")
    print("  Hottest day:", round(sel["temperature"].max(), 2), "C")
    print("  Coldest day:", round(sel["temperature"].min(), 2), "C")

print("
All years:")
for yr, avg in yearly.items():
    tag = ""
    if yr == best_year:     tag += " <- BEST"
    if yr == worst_year:    tag += " <- WORST"
    if yr == selected_year: tag += " <- SELECTED"
    print(" ", yr, ":", round(avg, 2), "C" + tag)
print("=" * 45)

## Step 4 — Save to CSV

In [ ]:
csv_file = site_name + "_temperature_data.csv"
daily.to_csv(csv_file, index=False)
print("Saved", len(daily), "rows to", csv_file)
daily.head()

## Step 5 — Plot

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 1, figsize=(18, 10))

ax1 = axes[0]
ax1.plot(daily["date"], daily["temperature"], linewidth=0.6, color="steelblue")
ax1.set_title("Daily Temperature - " + site_name, fontsize=14)
ax1.set_ylabel("Temperature (C)")
ax1.grid(True, alpha=0.3)
for yr, color, label in [(best_year,"red","Best"),(worst_year,"blue","Worst")]:
    yd = daily[daily["date"].dt.year == yr]
    ax1.axvspan(yd["date"].min(), yd["date"].max(), alpha=0.15, color=color, label=label+" "+str(yr))
ax1.legend()

ax2 = axes[1]
colors = []
for yr in yearly.index:
    if yr == best_year:       colors.append("red")
    elif yr == worst_year:    colors.append("blue")
    elif yr == selected_year: colors.append("orange")
    else:                     colors.append("steelblue")
ax2.bar(yearly.index, yearly.values, color=colors, alpha=0.8)
ax2.set_title("Yearly Average Temperature", fontsize=14)
ax2.set_xlabel("Year")
ax2.set_ylabel("Avg Temp (C)")
ax2.grid(True, alpha=0.3, axis="y")
ax2.legend(handles=[
    Patch(facecolor="red",       label="Best ("+str(best_year)+")"),
    Patch(facecolor="blue",      label="Worst ("+str(worst_year)+")"),
    Patch(facecolor="orange",    label="Selected ("+str(selected_year)+")"),
    Patch(facecolor="steelblue", label="Other years"),
])
plt.tight_layout()
plt.savefig(site_name + "_plot.png", dpi=150)
plt.show()
print("Plot saved!")